# Bluestock Mutual Fund Analytics Capstone - Day 2: Data Cleaning & Star Schema Database Load

## Day 2 Objectives:
1. **Clean all 10 raw CSV datasets** (handles nulls, duplicates, data types, date formatting).
2. **Handle weekends/holidays** in NAV and benchmark records using reindexing + forward-fill (`ffill()`).
3. **Compute derived fields** like `daily_return_pct` for NAV history.
4. **Standardize transactional fields** (types: SIP/Lumpsum/Redemption, KYC status: Verified/Pending).
5. Save all cleaned files to `data/processed/` folder as deliverables.

In [1]:
import os
import pandas as pd
import numpy as np
import re

# Paths
base_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
raw_dir = os.path.join(base_dir, "data", "raw")
processed_dir = os.path.join(base_dir, "data", "processed")
os.makedirs(processed_dir, exist_ok=True)
print(f"Raw: {raw_dir}\nProcessed: {processed_dir}")

Raw: X:\CODING\PROJECTS\InternshipWork\Bluestock\Capstone\data\raw
Processed: X:\CODING\PROJECTS\InternshipWork\Bluestock\Capstone\data\processed


### 1. Clean Fund Master (`01_fund_master.csv`)

In [2]:
print("Cleaning Fund Master...")
df_fund = pd.read_csv(os.path.join(raw_dir, "01_fund_master.csv"))
df_fund.columns = df_fund.columns.str.strip().str.lower()
df_fund['amfi_code'] = df_fund['amfi_code'].astype(str)
df_fund['launch_date'] = pd.to_datetime(df_fund['launch_date']).dt.strftime('%Y-%m-%d')
df_fund.to_csv(os.path.join(processed_dir, "01_fund_master_clean.csv"), index=False)
print(f"Saved Cleaned Fund Master: {df_fund.shape}")

Cleaning Fund Master...
Saved Cleaned Fund Master: (40, 15)


### 2. Clean NAV History (`02_nav_history.csv`)
We must sort by code + date, drop duplicate rows, validate `nav > 0`, and crucially, **reindex the dates to include weekends/holidays and forward-fill NAV**.

In [3]:
print("Cleaning NAV History...")
df_nav = pd.read_csv(os.path.join(raw_dir, "02_nav_history.csv"))
df_nav.columns = df_nav.columns.str.strip().str.lower()
df_nav['amfi_code'] = df_nav['amfi_code'].astype(str)
df_nav['date'] = pd.to_datetime(df_nav['date'])

# Exclude invalid NAV
df_nav = df_nav[df_nav['nav'] > 0]

# Drop exact duplicates
df_nav = df_nav.drop_duplicates(subset=['amfi_code', 'date'])

# Reindex per fund scheme to fill weekends/holidays
cleaned_navs = []
for code, group in df_nav.groupby('amfi_code'):
    group = group.set_index('date').sort_index()
    # Generate full daily date range
    full_idx = pd.date_range(start=group.index.min(), end=group.index.max(), freq='D')
    group = group.reindex(full_idx)
    group['amfi_code'] = code
    group['nav'] = group['nav'].ffill()
    # Compute daily return percent
    group['daily_return_pct'] = group['nav'].pct_change().fillna(0.0)
    group.index.name = 'date'
    cleaned_navs.append(group.reset_index())

df_nav_clean = pd.concat(cleaned_navs)
df_nav_clean['date'] = df_nav_clean['date'].dt.strftime('%Y-%m-%d')
df_nav_clean.to_csv(os.path.join(processed_dir, "02_nav_history_clean.csv"), index=False)
print(f"Saved Cleaned NAV History: {df_nav_clean.shape}")

Cleaning NAV History...


Saved Cleaned NAV History: (64320, 4)


### 3. Clean AMC AUM (`03_aum_by_fund_house.csv`)

In [4]:
print("Cleaning AUM by Fund House...")
df_aum = pd.read_csv(os.path.join(raw_dir, "03_aum_by_fund_house.csv"))
df_aum.columns = df_aum.columns.str.strip().str.lower()
df_aum['date'] = pd.to_datetime(df_aum['date']).dt.strftime('%Y-%m-%d')
df_aum.to_csv(os.path.join(processed_dir, "03_aum_by_fund_house_clean.csv"), index=False)
print(f"Saved Cleaned AUM: {df_aum.shape}")

Cleaning AUM by Fund House...
Saved Cleaned AUM: (90, 5)


### 4. Clean Monthly SIP Inflows (`04_monthly_sip_inflows.csv`)

In [5]:
print("Cleaning Monthly SIP Inflows...")
df_sip = pd.read_csv(os.path.join(raw_dir, "04_monthly_sip_inflows.csv"))
df_sip.columns = df_sip.columns.str.strip().str.lower()
df_sip.to_csv(os.path.join(processed_dir, "04_monthly_sip_inflows_clean.csv"), index=False)
print(f"Saved Cleaned SIP Inflows: {df_sip.shape}")

Cleaning Monthly SIP Inflows...
Saved Cleaned SIP Inflows: (48, 6)


### 5. Clean Category Inflows (`05_category_inflows.csv`)

In [6]:
print("Cleaning Category Inflows...")
df_cat = pd.read_csv(os.path.join(raw_dir, "05_category_inflows.csv"))
df_cat.columns = df_cat.columns.str.strip().str.lower()
df_cat.to_csv(os.path.join(processed_dir, "05_category_inflows_clean.csv"), index=False)
print(f"Saved Cleaned Category Inflows: {df_cat.shape}")

Cleaning Category Inflows...
Saved Cleaned Category Inflows: (144, 3)


### 6. Clean Industry Folio Counts (`06_industry_folio_count.csv`)

In [7]:
print("Cleaning Industry Folio Counts...")
df_folio = pd.read_csv(os.path.join(raw_dir, "06_industry_folio_count.csv"))
df_folio.columns = df_folio.columns.str.strip().str.lower()
df_folio.to_csv(os.path.join(processed_dir, "06_industry_folio_count_clean.csv"), index=False)
print(f"Saved Cleaned Folio Counts: {df_folio.shape}")

Cleaning Industry Folio Counts...
Saved Cleaned Folio Counts: (21, 6)


### 7. Clean Scheme Performance (`07_scheme_performance.csv`)
We need to validate returns are numeric, flag negative Sharpe ratios, and verify that expense ratios are within typical ranges (0.1% to 2.5%).

In [8]:
print("Cleaning Scheme Performance...")
df_perf = pd.read_csv(os.path.join(raw_dir, "07_scheme_performance.csv"))
df_perf.columns = df_perf.columns.str.strip().str.lower()
df_perf['amfi_code'] = df_perf['amfi_code'].astype(str)

# Replace missing morningstar ratings with a default value (e.g. 3)
if 'morningstar_rating' in df_perf.columns:
    df_perf['morningstar_rating'] = df_perf['morningstar_rating'].fillna(3).astype(int)

# Save
df_perf.to_csv(os.path.join(processed_dir, "07_scheme_performance_clean.csv"), index=False)
print(f"Saved Cleaned Scheme Performance: {df_perf.shape}")

Cleaning Scheme Performance...
Saved Cleaned Scheme Performance: (40, 19)


### 8. Clean Investor Transactions (`08_investor_transactions.csv`)
Standardize transaction types to `SIP`, `Lumpsum`, or `Redemption`. Validate `amount_inr > 0`. Verify `kyc_status` values.

In [9]:
print("Cleaning Investor Transactions...")
df_tx = pd.read_csv(os.path.join(raw_dir, "08_investor_transactions.csv"))
df_tx.columns = df_tx.columns.str.strip().str.lower()
df_tx['amfi_code'] = df_tx['amfi_code'].astype(str)
df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])

# Standardize transaction type
df_tx['transaction_type'] = df_tx['transaction_type'].str.strip().str.capitalize()
# Ensure only valid categories: SIP, Lumpsum, Redemption
df_tx['transaction_type'] = df_tx['transaction_type'].replace({'Sip': 'SIP', 'Redeem': 'Redemption', 'Lump': 'Lumpsum', 'Lumpsum': 'Lumpsum'})

# Validate amount
df_tx = df_tx[df_tx['amount_inr'] > 0]

# Standardize KYC Status
df_tx['kyc_status'] = df_tx['kyc_status'].str.strip().str.capitalize()

df_tx['transaction_date'] = df_tx['transaction_date'].dt.strftime('%Y-%m-%d')
df_tx.to_csv(os.path.join(processed_dir, "08_investor_transactions_clean.csv"), index=False)
print(f"Saved Cleaned Transactions: {df_tx.shape}")

Cleaning Investor Transactions...
Saved Cleaned Transactions: (32778, 13)


### 9. Clean Portfolio Holdings (`09_portfolio_holdings.csv`)

In [10]:
print("Cleaning Portfolio Holdings...")
df_port = pd.read_csv(os.path.join(raw_dir, "09_portfolio_holdings.csv"))
df_port.columns = df_port.columns.str.strip().str.lower()
df_port['amfi_code'] = df_port['amfi_code'].astype(str)
df_port.to_csv(os.path.join(processed_dir, "09_portfolio_holdings_clean.csv"), index=False)
print(f"Saved Cleaned Portfolio Holdings: {df_port.shape}")

Cleaning Portfolio Holdings...
Saved Cleaned Portfolio Holdings: (322, 8)


### 10. Clean Benchmark Indices (`10_benchmark_indices.csv`)
We must sort by date, drop duplicate rows, and reindex to daily range + forward-fill benchmark prices.

In [11]:
print("Cleaning Benchmark Indices...")
df_bench = pd.read_csv(os.path.join(raw_dir, "10_benchmark_indices.csv"))
df_bench.columns = df_bench.columns.str.strip().str.lower()
df_bench['date'] = pd.to_datetime(df_bench['date'])

# Sort and remove duplicates
df_bench = df_bench.sort_values('date').drop_duplicates(subset=['date'])

# Reindex daily to fill weekends/holidays
df_bench = df_bench.set_index('date')
full_idx = pd.date_range(start=df_bench.index.min(), end=df_bench.index.max(), freq='D')
df_bench = df_bench.reindex(full_idx)
df_bench = df_bench.ffill()
df_bench.index.name = 'date'
df_bench = df_bench.reset_index()

df_bench['date'] = df_bench['date'].dt.strftime('%Y-%m-%d')
df_bench.to_csv(os.path.join(processed_dir, "10_benchmark_indices_clean.csv"), index=False)
print(f"Saved Cleaned Benchmark Indices: {df_bench.shape}")

Cleaning Benchmark Indices...
Saved Cleaned Benchmark Indices: (1608, 3)


### Final Status Check
Verify that all 10 cleaned CSV files have been created in `data/processed/`.

In [12]:
print("Cleaned CSV Deliverables:")
for f in sorted(os.listdir(processed_dir)):
    if f.endswith('.csv'):
        fp = os.path.join(processed_dir, f)
        print(f"- {f}: {os.path.getsize(fp)/1024:.2f} KB")

Cleaned CSV Deliverables:
- 01_fund_master_clean.csv: 6.70 KB
- 02_nav_history_clean.csv: 2773.56 KB
- 03_aum_by_fund_house_clean.csv: 4.01 KB
- 04_monthly_sip_inflows_clean.csv: 1.68 KB
- 05_category_inflows_clean.csv: 3.72 KB
- 06_industry_folio_count_clean.csv: 0.82 KB
- 07_scheme_performance_clean.csv: 6.48 KB
- 08_investor_transactions_clean.csv: 3086.62 KB
- 09_portfolio_holdings_clean.csv: 23.66 KB
- 10_benchmark_indices_clean.csv: 50.50 KB
